<a href="https://colab.research.google.com/github/thisisaadi123/chronos-project/blob/main/Chronos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multivariate Retail Forecasting via Foundation Models
## Causal Inference on Intermittent Demand Sequences

### Abstract
Forecasting intermittent retail demand presents a significant challenge for traditional statistical frameworks due to high sparsity and non-stationary variance. This project implements a zero-shot forecasting pipeline leveraging the Chronos-T5 transformer architecture. By aligning target sales sequences with dynamic pricing and temporal covariates, the model generates probabilistic risk envelopes (P10–P90) used for inventory optimization.

### Technical Implementation
* Data Engineering: Horizontal-to-vertical unpivoting (Melting) and multi-source feature alignment.
* Normalization: Localized standardization followed by Inverse Hyperbolic Sine (arcsinh) scaling to handle 0-value density and promotional outliers.
* Architecture: 3D Tensor formulation for Alternating Intra/Inter-series Attention.
* Evaluation: Calibration testing via Mean Weighted Quantile Loss (wQL) to assess probability envelope accuracy.

### Raw Data Architecture (Data Dictionary)

Before manipulating the data, we must understand the schema of the three core tables provided by Walmart. To build a multivariate causal model, we will need to extract and join specific columns of interest across these tables.

#### 1. Sales Data (`sales_raw`)
This table records the daily unit sales per product. It is formatted as a "Wide" dataframe.
* `id` *(string)*: Unique identifier combining the item and store. **[Target Key]**
* `item_id`, `dept_id`, `cat_id` *(string)*: Product hierarchy metadata.
* `store_id`, `state_id` *(string)*: Geographical metadata.
* `d_1` to `d_1941` *(integer)*: The exact number of units sold on day $d$. **[Primary Target Variable]**

#### 2. Calendar Data (`calendar_raw`)
This table maps the day indices (`d_1`) to real-world dates, weeks, and special events.
* `date` *(string)*: Standard YYYY-MM-DD format.
* `wm_yr_wk` *(integer)*: Walmart's proprietary weekly ID. **[Join Key for Pricing]**
* `weekday`, `wday`, `month`, `year`: Standard temporal features.
* `d` *(string)*: The day index (e.g., "d_1") matching the sales columns. **[Join Key for Sales]**
* `event_name_1`, `event_type_1` *(string/NaN)*: Cultural/national holidays (e.g., Super Bowl, Christmas). **[Covariate of Interest]**
* `snap_CA`, `snap_TX`, `snap_WI` *(binary)*: Indicates if SNAP (food stamps) purchases were allowed that day.

#### 3. Pricing Data (`prices_raw`)
This table logs the historical weekly price of every item in every store.
* `store_id` *(string)*: Store location. **[Join Key]**
* `item_id` *(string)*: The specific product. **[Join Key]**
* `wm_yr_wk` *(integer)*: Walmart's weekly ID. **[Join Key]**
* `sell_price` *(float)*: The retail price of the item for that specific week. **[Covariate of Interest]**

**Objective:** We must isolate a specific `id`'s sales history, melt it into a vertical sequence, and map the `event_name_1` and `sell_price` to that timeline to create our final `[Batch, Variates, Length]` tensor.

In [30]:
import os
import pandas as pd

print("[INFO] Authenticating and downloading M5 dataset...")
os.environ['KAGGLE_API_TOKEN'] = "KGAT_a66b802dffecdfd4512d0a850e39d710"
!kaggle competitions download -c m5-forecasting-accuracy
!unzip -o -q m5-forecasting-accuracy.zip

print("Loading raw dataframes")
#load dataset
sales_raw = pd.read_csv('sales_train_evaluation.csv', nrows=50)
calendar_raw = pd.read_csv('calendar.csv')
prices_raw = pd.read_csv('sell_prices.csv')

print("Data Loaded")

[INFO] Authenticating and downloading M5 dataset...
m5-forecasting-accuracy.zip: Skipping, found more recently modified local copy (use --force to force download)
Loading raw dataframes
Data Loaded


### Phase 1: Sequence Formatting (The "Melt")

**The Concept of "Melting":**

In data science, datasets often arrive in a "Wide" format where time expands horizontally (e.g., Day 1, Day 2, and Day 3 are separate columns). While this is easy for humans to read in a spreadsheet, it breaks time-series AI models. Neural networks cannot read left-to-right across columns; they require a continuous, top-to-bottom sequence.

"Melting" (or unpivoting) is the programmatic process of crushing those horizontal columns into a "Long" format. We take all 1,941 day columns and compress them into just two clean, vertical columns: `day` and `sales`.

**Our Objective:**
We will use the pandas `melt` function to transform the raw Walmart data into a sequential timeline. Once the data flows vertically, we will isolate a single Hobby item to serve as the primary target variable for our foundation model.
### Phase 1: Executing the "Melt" (Target Extraction)

Now that we understand the Wide format, we will use pandas to crush the 1,941 day columns into a vertical timeline.

Once the data is vertical, we will isolate a single product (our "Target Item") to serve as the baseline for our foundation model. We will print the dataframe before and after this process so the transformation is clearly visible.

In [31]:
#define ids
id_vars = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
sales_long = sales_raw.melt(id_vars=id_vars, var_name='day_string', value_name='sales')
sales_long['day'] = sales_long['day_string'].str.replace('d_', '').astype(int)

#sort data
sales_long = sales_long.sort_values(['id', 'day']).drop(columns=['day_string'])

#isolate target item
target_item = sales_long['id'].iloc[0]


item_story = sales_long[sales_long['id'] == target_item].copy()

print(f"Target: {target_item}")
display(item_story[['id', 'day', 'sales']].head(5))

Target: HOBBIES_1_001_CA_1_evaluation


,id,day,sales
0,HOBBIES_1_001_CA_1_evaluation,1,0
50,HOBBIES_1_001_CA_1_evaluation,2,0
100,HOBBIES_1_001_CA_1_evaluation,3,0
150,HOBBIES_1_001_CA_1_evaluation,4,0
200,HOBBIES_1_001_CA_1_evaluation,5,0


### Phase 2: Engineering Causal Covariates

Time-series forecasting without covariates is just guessing. To predict *when* a spike will occur, the model needs to know why historical spikes occurred.

We will engineer two external variables (Covariates):
1. **Events (Temporal Covariate):** We will convert the Calendar's text events (e.g., "Thanksgiving") into a binary flag (1 for event, 0 for normal day).
2. **Prices (Dynamic Covariate):** We will map the weekly price of the item onto our daily timeline.

In [32]:
#processing calendar csv file
calendar_subset = calendar_raw[['d', 'event_name_1', 'wm_yr_wk']].copy()
calendar_subset['day'] = calendar_subset['d'].str.replace('d_', '').astype(int)

#1 if event 0 if no event
calendar_subset['event_flag'] = calendar_subset['event_name_1'].notna().astype(int)


item_df = pd.merge(item_story, calendar_subset[['day', 'wm_yr_wk', 'event_flag']], on='day', how='left')


item_prices = prices_raw[(prices_raw['item_id'] == item_df['item_id'].iloc[0]) &
                         (prices_raw['store_id'] == item_df['store_id'].iloc[0])]

item_df = pd.merge(item_df, item_prices[['wm_yr_wk', 'sell_price']], on='wm_yr_wk', how='left')

#missing data handled
item_df['sell_price'] = item_df['sell_price'].bfill().ffill()

print("Covariates are now connected")
display(item_df[['day', 'sales', 'event_flag', 'sell_price']].tail(5))

Covariates are now connected


,day,sales,event_flag,sell_price
1936,1937,0,0,8.38
1937,1938,3,0,8.38
1938,1939,3,0,8.38
1939,1940,0,0,8.38
1940,1941,1,0,8.38


### Phase 3: Visualizing the "Causal Tangle"

Before applying AI, we must visually verify the problem we are trying to solve. We will plot our Target (Sales) against our Covariate (Price).

Observe the extreme sparsity—days of zero sales followed by sudden spikes. More importantly, observe the pricing behavior: if the price suddenly drops or increases, does it alter the frequency of the sales spikes? This complex relationship is exactly what Chronos-T5 is designed to decode.

In [33]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

#visualizing using plotly
fig = make_subplots(specs=[[{"secondary_y": True}]])
plot_data = item_df.tail(200) # We only plot the last 200 days so the spikes are visible.

#plotting data
fig.add_trace(go.Scatter(x=plot_data['day'], y=plot_data['sales'],
                         name="Daily Sales", line=dict(color='#111')), secondary_y=False)

fig.add_trace(go.Scatter(x=plot_data['day'], y=plot_data['sell_price'],
                         name="Sell Price ($)", line=dict(color='#0066FF', dash='dot')), secondary_y=True)

#rendering data
fig.update_layout(title="Retail Causality: Price vs. Demand", template="plotly_white", height=400)
fig.show()

### Phase 4: Robust Scaling & Tensor Formatting

**The Math ($\sinh^{-1}$):**
Foundation models struggle with extreme variance. If a product usually sells 0 units but spikes to 20 during an event, that sudden "20" acts like a megaphone, blinding the model's attention mechanism. To fix this, we apply an **Inverse Hyperbolic Sine ($\sinh^{-1}$)** transformation. This elegantly squashes extreme outliers while safely handling the zeros (unlike a standard logarithm, which breaks on zeros).

**The Architecture:**
Chronos requires inputs in a highly specific mathematical shape: a 3-Dimensional PyTorch Tensor structured as `[Batch, Variates, Sequence Length]`.

In [34]:
import torch
import numpy as np


def chronos_scaler(series):
    mu, sigma = series.mean(), series.std() + 1e-8
    z_score = (series - mu) / sigma
    squashed = np.arcsinh(z_score)
    return squashed, mu, sigma


y_scaled, y_mu, y_sigma = chronos_scaler(item_df['sales'])
p_scaled, p_mu, p_sigma = chronos_scaler(item_df['sell_price'])
e_scaled, e_mu, e_sigma = chronos_scaler(item_df['event_flag'])

#constants
CONTEXT_LENGTH = 512
PREDICTION_LENGTH = 28

context_y = torch.tensor(y_scaled.values[-CONTEXT_LENGTH:]).float()
context_p = torch.tensor(p_scaled.values[-CONTEXT_LENGTH:]).float()
context_e = torch.tensor(e_scaled.values[-CONTEXT_LENGTH:]).float()


context_tensor = torch.stack([context_y, context_p, context_e])
print(f"Tensor ready with Shape: {context_tensor.shape}")

Tensor ready with Shape: torch.Size([3, 512])


### Phase 5: Zero-Shot Chronos Inference

We initialize the `amazon/chronos-t5-base` pipeline. Because Chronos is a Foundation Model, we **do not** train it. Instead, we control its performance by tuning its generation hyperparameters:
* `num_samples=50`: Generates 50 distinct future paths to build a robust probability curve.
* `temperature=0.8`: Reduces the randomness of the model to prevent hallucinated spikes.
* `top_p=0.9`: Prevents the model from exploring extreme outlier probabilities.

In [35]:
#install if necessary:!pip install git+https://github.com/amazon-science/chronos-forecasting.git
from chronos import ChronosPipeline
print("Loading Foundation Model to GPU")
pipeline = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-base",
    device_map="cuda" if torch.cuda.is_available() else "cpu",
    torch_dtype=torch.bfloat16,
)


print("Generating Probabilistic Forecast Paths...")
forecast = pipeline.predict(
    context_tensor,
    prediction_length=PREDICTION_LENGTH,
    num_samples=50,
    temperature=0.8,
    top_p=0.9
)

target_forecast = forecast[0].numpy()
low, median, high = np.quantile(target_forecast, [0.1, 0.5, 0.9], axis=0)
print("P10-P90 Envelope generated.")

Loading Foundation Model to GPU
Generating Probabilistic Forecast Paths...
P10-P90 Envelope generated.


### Phase 6: Calibration Evaluation (wQL)

Traditional metrics like Mean Absolute Error (MAE) are useless here—predicting a flat "zero" often yields the lowest MAE on sparse data, which guarantees stockouts for a retailer.

Instead, we use **Mean Weighted Quantile Loss (wQL)**. This metric evaluates the entire "Cone of Uncertainty." It heavily penalizes the model if actual sales fall outside the P10-P90 risk envelope, ensuring our model is properly calibrated for safety stock planning. A score under 1.0 is considered highly viable.

In [36]:
print("Calculating Weighted Quantile Loss (wQL)")
actual_truth = item_df['sales'].values[-PREDICTION_LENGTH:]

#wql function
def calculate_wql(actual, forecast_quantile, quantile_level):
    # Calculating the pinball loss
    diff = actual - forecast_quantile
    loss = np.maximum(quantile_level * diff, (quantile_level - 1) * diff)

    #Normalizing
    denominator = np.sum(np.abs(actual))
    return 2 * np.sum(loss) / denominator if denominator > 0 else 0


wql_p10 = calculate_wql(actual_truth, low, 0.10)
wql_p50 = calculate_wql(actual_truth, median, 0.50)
wql_p90 = calculate_wql(actual_truth, high, 0.90)

mean_wql = (wql_p10 + wql_p50 + wql_p90) / 3

#output
print(f"P10 wQL (Stockout Risk): {wql_p10:.4f}")
print(f"P50 wQL (Median Error):  {wql_p50:.4f}")
print(f"P90 wQL (Overstock Risk):{wql_p90:.4f}")
print("-" * 30)
print(f"Mean wQL Score:          {mean_wql:.4f}")

Calculating Weighted Quantile Loss (wQL)
P10 wQL (Stockout Risk): 0.6815
P50 wQL (Median Error):  0.9488
P90 wQL (Overstock Risk):0.8832
------------------------------
Mean wQL Score:          0.8378
